In [1]:
import pandas as pd
import numpy as np


In [2]:
customers=pd.read_excel(r'C:\Users\suraj\Downloads\ecommerce_dataset.xlsx',sheet_name= 'Customers')
orders=pd.read_excel(r'C:\Users\suraj\Downloads\ecommerce_dataset.xlsx',sheet_name= 'Orders')
products=pd.read_excel(r'C:\Users\suraj\Downloads\ecommerce_dataset.xlsx',sheet_name= 'Products')
returns=pd.read_excel(r'C:\Users\suraj\Downloads\ecommerce_dataset.xlsx',sheet_name= 'Returns')
activity=pd.read_excel(r'C:\Users\suraj\Downloads\ecommerce_dataset.xlsx',sheet_name= 'Customer_Activity')
customers.head()



,Customer_ID,Signup_Date,Gender,Age,City,Region,Acquisition_Channel,Customer_Segment
0,CUST00001,2025-08-13,Male,56.0,Chennai,South,Email,VIP
1,CUST00002,2023-10-10,Female,46.0,Bengaluru,South,Referral,Standard
2,CUST00003,2024-07-10,Male,NaN,Jaipur,North,Meta,Premium
3,CUST00004,2024-05-14,Male,60.0,Mumbai,West,Referral,VIP
4,CUST00005,2025-11-28,Female,25.0,Mumbai,West,Email,Standard


In [3]:
def missing_value_report(df,sheet_name):
    total_row= len(df)
    null_counts= df.isnull().sum()
    null_percentage= (null_counts/total_row*100).round(2)

    report= pd.DataFrame({ "column": null_counts.index,
                          "missing_counts":null_counts.values,
                          "missing_percentage":null_percentage.values
    })

    report= report[report['missing_counts']>0].reset_index(drop=True)
    return report

In [4]:
customers_report = missing_value_report(customers, "Customers")
orders_report = missing_value_report(orders, "Orders")
products_report = missing_value_report(products, "Products")
returns_report = missing_value_report(returns, "Returns")
activity_report = missing_value_report(activity, "Customer_Activity")


In [5]:
master_report = pd.concat([
    customers_report.assign(Sheet="Customers"),
    orders_report.assign(Sheet="Orders"),
    products_report.assign(Sheet="Products"),
    returns_report.assign(Sheet="Returns"),
    activity_report.assign(Sheet="Customer_Activity")
], ignore_index=True)

print("===== MASTER SUMMARY =====")
print(master_report)

===== MASTER SUMMARY =====
                 column  missing_counts  missing_percentage              Sheet
0                Gender              91                2.94          Customers
1                   Age             123                3.98          Customers
2                  City              95                3.07          Customers
3   Acquisition_Channel             125                4.05          Customers
4      Customer_Segment              62                2.01          Customers
5              Quantity             309                2.00             Orders
6              Discount             614                3.97             Orders
7        Payment_Method             458                2.96             Orders
8          Sub_Category              17                5.56           Products
9            Cost_Price               9                2.94           Products
10        Return_Reason              91                5.89            Returns
11        Refund_Amount  

In [6]:
master_report.to_csv("_missing_value_summary.csv",index=False)
print("Saved successfully!")
print(master_report)

Saved successfully!
                 column  missing_counts  missing_percentage              Sheet
0                Gender              91                2.94          Customers
1                   Age             123                3.98          Customers
2                  City              95                3.07          Customers
3   Acquisition_Channel             125                4.05          Customers
4      Customer_Segment              62                2.01          Customers
5              Quantity             309                2.00             Orders
6              Discount             614                3.97             Orders
7        Payment_Method             458                2.96             Orders
8          Sub_Category              17                5.56           Products
9            Cost_Price               9                2.94           Products
10        Return_Reason              91                5.89            Returns
11        Refund_Amount         

In [7]:
def duplicate_report(df, id_column, sheet_name):
    total_rows= len(df)
    full_row_duplicates= df.duplicated().sum()
    id_duplicates= df[id_column].duplicated().sum()

    print(f"---{sheet_name}---")
    print(f"total_rows:{total_rows}")
    print(f"full duplicate rows:{full_row_duplicates}")
    print(f"duplicate customer_id:{id_duplicates}")
    print()

duplicate_report(customers,"Customer_ID","Customers")
duplicate_report(products, "Product_ID", "Products")
duplicate_report(orders, "Order_ID", "Orders")
duplicate_report(returns, "Return_ID", "Returns")
duplicate_report(activity, "Customer_ID", "Customer_Activity")


---Customers---
total_rows:3090
full duplicate rows:90
duplicate customer_id:90

---Products---
total_rows:306
full duplicate rows:6
duplicate customer_id:6

---Orders---
total_rows:15450
full duplicate rows:450
duplicate customer_id:450

---Returns---
total_rows:1545
full duplicate rows:45
duplicate customer_id:45

---Customer_Activity---
total_rows:31200
full duplicate rows:1200
duplicate customer_id:28200



In [8]:
print("--- Customers dtypes ---")
print(customers.dtypes)
print()

print("--- Products dtypes ---")
print(products.dtypes)
print()

print("--- Orders dtypes ---")
print(orders.dtypes)
print()

print("--- Returns dtypes ---")
print(returns.dtypes)
print()

print("--- Customer_Activity dtypes ---")
print(activity.dtypes)

--- Customers dtypes ---
Customer_ID                    object
Signup_Date            datetime64[ns]
Gender                         object
Age                           float64
City                           object
Region                         object
Acquisition_Channel            object
Customer_Segment               object
dtype: object

--- Products dtypes ---
Product_ID        object
Product_Name      object
Category          object
Sub_Category      object
Cost_Price       float64
Selling_Price    float64
dtype: object

--- Orders dtypes ---
Order_ID                  object
Customer_ID               object
Order_Date        datetime64[ns]
Product_ID                object
Quantity                 float64
Unit_Price               float64
Discount                 float64
Payment_Method            object
dtype: object

--- Returns dtypes ---
Return_ID                object
Order_ID                 object
Return_Date      datetime64[ns]
Return_Reason            object
Refund_Amount  

In [9]:
print("negative age",(customers['Age']<0).sum())
print("usually high age",(customers['Age']>100).sum())

negative age 0
usually high age 0


In [10]:
# --- Products: Price checks ---
print("Negative Cost_Price:", (products['Cost_Price'] < 0).sum())
print("Negative Selling_Price:", (products['Selling_Price'] < 0).sum())
print("Cost_Price > Selling_Price (loss-making):", (products['Cost_Price'] > products['Selling_Price']).sum())
print()

Negative Cost_Price: 0
Negative Selling_Price: 0
Cost_Price > Selling_Price (loss-making): 0



In [11]:
# --- Orders: Quantity, Price, Discount checks ---
print("negative quantitiy:",(orders['Quantity']<0).sum())
print("zero quantitiy:",(orders['Quantity']==0).sum())
print("negative uinit price:",(orders['Unit_Price']==0).sum())
print("Discount > 1 (i.e. more than 100%):", (orders['Discount'] > 1).sum())
print("Negative Discount:", (orders['Discount'] < 0).sum())
print()


negative quantitiy: 0
zero quantitiy: 0
negative uinit price: 0
Discount > 1 (i.e. more than 100%): 0
Negative Discount: 0



In [12]:
# --- Returns: Refund_Amount check ---
print("Negative Refund_Amount:", (returns['Refund_Amount'] < 0).sum())

Negative Refund_Amount: 0


In [13]:
 print("----Customers table-----")
print()
print("Gender unique values:", customers['Gender'].unique())
print()
print("City unique values:", customers['City'].unique())
print()
print("Region unique values:", customers['Region'].unique())
print()
print("Acquisition_Channel unique values:", customers['Acquisition_Channel'].unique())
print()
print("Customer_Segment unique values:", customers['Customer_Segment'].unique())
print()

 print("----products table-----")
print()
print("Category unique values:", products['Category'].unique())
print()
print("Sub_Category unique values:", products['Sub_Category'].unique())
print()

 print("----orders table-----")
print()
print("Payment_Method unique values:", orders['Payment_Method'].unique())
print()

 print("----returns table-----")
print()
print("Return_Reason unique values:", returns['Return_Reason'].unique())
print()

 print("----activity table-----")
print()
print("App_Usage unique values:", activity['App_Usage'].unique())

----Customers table-----

Gender unique values: ['Male' 'Female' nan]

City unique values: ['Chennai' 'Bengaluru' 'Jaipur' 'Mumbai' 'Hyderabad' 'Pune' 'Kolkata'
 'Delhi' 'Noida' 'Gurugram' nan]

Region unique values: ['South' 'North' 'West' 'East']

Acquisition_Channel unique values: ['Email' 'Referral' 'Meta' 'Organic' 'Google' 'Affiliate' nan]

Customer_Segment unique values: ['VIP' 'Standard' 'Premium' nan]

----products table-----

Category unique values: ['Electronics' 'Sports' 'Beauty' 'Fashion' 'Home']

Sub_Category unique values: ['Wearables' 'Team Sports' 'Haircare' 'Accessories' 'Kitchen'
 'Personal Care' 'Audio' 'Fitness' 'Mobile Accessories' 'Cameras'
 'Skincare' 'Men' 'Makeup' 'Women' nan 'Decor' 'Footwear' 'Outdoor' 'Yoga'
 'Bedding' 'Furniture']

----orders table-----

Payment_Method unique values: ['UPI' 'Net Banking' 'Credit Card' 'COD' 'Debit Card' nan]

----returns table-----

Return_Reason unique values: ['Defective Product' 'Wrong Item' 'Late Delivery' 'Changed Min

In [14]:
invalid_customers_in_orders= orders[~orders['Customer_ID'].isin( customers['Customer_ID'])]
print("Orders with invalid Customer_ID:",len(invalid_customers_in_orders))

Orders with invalid Customer_ID: 0


In [15]:
invalid_products_in_orders= orders[~orders['Product_ID'].isin(products['Product_ID'])]
print("Orders with invalid Product_ID:",len(invalid_products_in_orders))

Orders with invalid Product_ID: 0


In [16]:
invalid_orders_in_returns = returns[~returns['Order_ID'].isin(orders['Order_ID'])]
print("Returns with invalid Order_ID:", len(invalid_orders_in_returns))

Returns with invalid Order_ID: 0


In [17]:
invalid_customers_in_activity = activity[~activity['Customer_ID'].isin(customers['Customer_ID'])]
print("Activity with invalid Customer_ID:", len(invalid_customers_in_activity))

Activity with invalid Customer_ID: 0


In [18]:
#  Return_Date, Order_Date ke baad honi chahiye
merge_return=returns.merge(orders[['Order_ID', 'Order_Date']],on='Order_ID',how='left')
invalid_return_dates= merge_return[merge_return['Return_Date']< merge_return['Order_Date']]
print("Returns where Return_Date is BEFORE Order_Date:", len(invalid_return_dates))
print()

#  Order_Date, Customer ki Signup_Date ke baad honi chahiye
merged_orders = orders.merge(customers[['Customer_ID', 'Signup_Date']], on='Customer_ID', how='left')
invalid_order_dates = merged_orders[merged_orders['Order_Date'] < merged_orders['Signup_Date']]
print("Orders placed BEFORE customer signed up:", len(invalid_order_dates))

Returns where Return_Date is BEFORE Order_Date: 0

Orders placed BEFORE customer signed up: 5379


In [27]:
def find_outliers(df,column):
    Q1= df[column].quantile(0.25)
    Q3= df[column].quantile(0.75)
    IQR= Q3-Q1

    Lower_bound= Q1-1.5*IQR
    Upper_bound= Q3+1.5*IQR

    outliers= df[(df[column]<Lower_bound) | (df[column]>Upper_bound)]

    print(f"column:{column}")
    print(f"normal range: {Lower_bound:.2f} to {Upper_bound:.2f}")
    print(f"Outliers found: {len(outliers)}")
    print()
    
    return outliers
    

In [29]:
# Orders table pe check karo
find_outliers(orders, 'Unit_Price')
find_outliers(orders, 'Quantity')

# Products table pe check karo
find_outliers(products, 'Cost_Price')
find_outliers(products, 'Selling_Price')

# Returns table pe check karo
find_outliers(returns, 'Refund_Amount')

# Customers table pe check karo
find_outliers(customers, 'Age')


column:Unit_Price
normal range: -3396.72 to 10626.28
Outliers found: 0

column:Quantity
normal range: -3.50 to 8.50
Outliers found: 0

column:Cost_Price
normal range: -2362.86 to 7494.42
Outliers found: 0

column:Selling_Price
normal range: -3214.95 to 10546.06
Outliers found: 0

column:Refund_Amount
normal range: -1693.56 to 5948.81
Outliers found: 0

column:Age
normal range: -3.75 to 82.25
Outliers found: 0



,Customer_ID,Signup_Date,Gender,Age,City,Region,Acquisition_Channel,Customer_Segment


In [30]:
print("=" * 60)
print("FINAL DATA VALIDATION SUMMARY")
print("=" * 60)

print("\n1. Missing Values (Step 3.1)")
print("   → Present in multiple columns (see Step 3.1 report)")

print("\n2. Duplicates (Step 3.2)")
print(f"   Customers duplicate rows: {customers.duplicated().sum()}")
print(f"   Products duplicate rows: {products.duplicated().sum()}")
print(f"   Orders duplicate rows: {orders.duplicated().sum()}")
print(f"   Returns duplicate rows: {returns.duplicated().sum()}")
print(f"   Customer_Activity duplicate rows: {activity.duplicated().sum()}")

print("\n3. Data Types (Step 3.3)")
print("   → All columns verified — dates as datetime, numerics as float/int")

print("\n4. Invalid Values (Step 3.4)")
print("   → No negative Age, Price, Quantity, or invalid Discount found")

print("\n5. Categorical Consistency (Step 3.5)")
print("   → All category values are clean and standardized")

print("\n6. Referential Integrity (Step 3.6)")
print("   → All Customer_ID, Product_ID, Order_ID relationships are valid")

print("\n7. Business Rule Violations (Step 3.7)")
print(f"   Orders placed before Signup_Date: 5379  ⚠️ (needs fixing)")
print(f"   Returns before Order_Date: 0")

print("\n8. Outliers (Step 3.8)")
print("   → See individual column results above")

print("\n" + "=" * 60)
print("STATUS: Dataset has identified issues that need cleaning")
print("        before proceeding to EDA (Step 4)")
print("=" * 60)

FINAL DATA VALIDATION SUMMARY

1. Missing Values (Step 3.1)
   → Present in multiple columns (see Step 3.1 report)

2. Duplicates (Step 3.2)
   Customers duplicate rows: 90
   Products duplicate rows: 6
   Orders duplicate rows: 450
   Returns duplicate rows: 45
   Customer_Activity duplicate rows: 1200

3. Data Types (Step 3.3)
   → All columns verified — dates as datetime, numerics as float/int

4. Invalid Values (Step 3.4)
   → No negative Age, Price, Quantity, or invalid Discount found

5. Categorical Consistency (Step 3.5)
   → All category values are clean and standardized

6. Referential Integrity (Step 3.6)
   → All Customer_ID, Product_ID, Order_ID relationships are valid

7. Business Rule Violations (Step 3.7)
   Orders placed before Signup_Date: 5379  ⚠️ (needs fixing)
   Returns before Order_Date: 0

8. Outliers (Step 3.8)
   → See individual column results above

STATUS: Dataset has identified issues that need cleaning
        before proceeding to EDA (Step 4)


In [31]:
customers = customers.drop_duplicates()
products = products.drop_duplicates()
orders = orders.drop_duplicates()
returns = returns.drop_duplicates()
activity = activity.drop_duplicates()

print("Duplicates removed. New row counts:")
print("Customers:", len(customers))
print("Products:", len(products))
print("Orders:", len(orders))
print("Returns:", len(returns))
print("Activity:", len(activity))

Duplicates removed. New row counts:
Customers: 3000
Products: 300
Orders: 15000
Returns: 1500
Activity: 30000


In [38]:
# --- Customers ---
customers['Age']= customers['Age'].fillna(customers['Age'].median())
customers['Gender'] = customers['Gender'].fillna('Unknown')
customers['City'] = customers['City'].fillna('Unknown')
customers['Acquisition_Channel'] = customers['Acquisition_Channel'].fillna('Unknown')
customers['Customer_Segment'] = customers['Customer_Segment'].fillna('Unknown')
# --- Products ---
products['Sub_Category'] = products['Sub_Category'].fillna('Unknown')
products['Cost_Price'] = products['Cost_Price'].fillna(products['Cost_Price'].median())

# --- Orders ---
orders['Quantity'] = orders['Quantity'].fillna(orders['Quantity'].median())
orders['Discount'] = orders['Discount'].fillna(0)   # missing discount = assume no discount
orders['Payment_Method'] = orders['Payment_Method'].fillna('Unknown')

# --- Returns ---
returns['Return_Reason'] = returns['Return_Reason'].fillna('Unknown')
returns['Refund_Amount'] = returns['Refund_Amount'].fillna(returns['Refund_Amount'].median())

# --- Customer_Activity ---
activity['Session_Minutes'] = activity['Session_Minutes'].fillna(activity['Session_Minutes'].median())
activity['Support_Tickets'] = activity['Support_Tickets'].fillna(0)
activity['App_Usage'] = activity['App_Usage'].fillna('Unknown')

print("Missing values handled.")

Missing values handled.


C:\Users\suraj\AppData\Local\Temp\ipykernel_4600\1611706459.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders['Quantity'] = orders['Quantity'].fillna(orders['Quantity'].median())
C:\Users\suraj\AppData\Local\Temp\ipykernel_4600\1611706459.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders['Discount'] = orders['Discount'].fillna(0)   # missing discount = assume no discount
C:\Users\suraj\AppData\Local\Temp\ipykernel_4600\1611706459.py:14: SettingWithCopyWarning: 
A value is trying to be se

In [41]:
# invalid Order_IDs 
merged_check = orders.merge(customers[['Customer_ID', 'Signup_Date']], on='Customer_ID', how='left')
invalid_ids = merged_check[merged_check['Order_Date'] < merged_check['Signup_Date']]['Order_ID']

# Orders table se hatao
orders = orders[~orders['Order_ID'].isin(invalid_ids)]

#  Returns table se  hatao 
returns = returns[~returns['Order_ID'].isin(invalid_ids)]

print("Invalid orders removed:", len(invalid_ids))
print("Orders remaining:", len(orders))
print("Returns remaining (after removing related invalid orders):", len(returns))

Invalid orders removed: 5079
Orders remaining: 9921
Returns remaining (after removing related invalid orders): 962


In [49]:
print("\n=== FINAL CHECK ===")
print("Total nulls remaining:")
print("Customers:", customers.isnull().sum().sum())
print("Products:", products.isnull().sum().sum())
print("Orders:", orders.isnull().sum().sum())
print("Returns:", returns.isnull().sum().sum())
print("Activity:", activity.isnull().sum().sum())


=== FINAL CHECK ===
Total nulls remaining:
Customers: 0
Products: 0
Orders: 0
Returns: 0
Activity: 0


In [55]:
with pd.ExcelWriter('ecommerce_cleaned_data.xlsx') as writer:
    customers.to_excel(writer, sheet_name='Customers', index=False)
    products.to_excel(writer, sheet_name='Products', index=False)
    orders.to_excel(writer, sheet_name='Orders', index=False)
    returns.to_excel(writer, sheet_name='Returns', index=False)
    activity.to_excel(writer, sheet_name='Customer_Activity', index=False)

print("Cleaned dataset saved!")

Cleaned dataset saved!
